Joyce Wang

02/17/2026

# Gene Set Enrichment Analysis with GSEApy: Head-to-Head Comparisons

In [1]:
# Core libraries
import hisepy
import numpy as np
import scanpy as sc
import anndata as ad
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
import seaborn as sns

from gseapy import dotplot

/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:413: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\frac{1}{m} \\sum_{ij} \\left(A_{ij} - \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/VertexPartition.py:788: SyntaxWarning: invalid escape sequence '\m'
  .. math:: Q = \\sum_{ij} \\left(A_{ij} - \\gamma \\frac{k_i^\mathrm{out} k_j^\mathrm{in}}{m} \\right)\\delta(\\sigma_i, \\sigma_j),
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:27: SyntaxWarning: invalid escape sequence '\g'
  implementation therefore does not guarantee subpartition :math:`\gamma`-density.
/home/workspace/environment/pythonscrna12/lib/python3.13/site-packages/leidenalg/Optimiser.py:346: SyntaxWarning: invalid escape sequence '\s'
  .. math:: Q = \sum_k \\lambda_k Q_k.


**Note:** In the drug treatment vs. DMSO control GSEA notebooks (`gseapy_Hallmark_JW.ipynb`, `gseapy_KEGG_JW.ipynb`, `gseapy_Reactome_JW.ipynb`), I manually read in the data, cleaned up the labels, normalized the data, and performed DGE for the purpose of showing the steps of differential gene expression analysis in scanpy. **These results are the same as Sean's DEG results csv files uploaded onto HISE.** In the head-to-head formulation comparisons GSEA notebooks (`gseapy_head_to_head_Hallmark_JW.ipynb`, `gseapy_head_to_head_KEGG_JW.ipynb`, `gseapy_head_to_head_Reactome_JW.ipynb`), I show the process of loading in the DEG results csv from HISE and performed GSEA after.

# Load in DEG Results for head-to-head comparisons between formulations for each cell type

In [2]:
file_id = ['5a715df8-b774-4947-a023-5fd9c9d97e19']

# dict
file_path = hisepy.read_files(file_list=file_id)

In [3]:
# dict values --> list
deg_dfs = list(file_path.values())

In [4]:
# concatenate
deg_df = pd.concat(deg_dfs, ignore_index=True)

deg_df

,projectGuid,mergeKey,emr,labLastModified,lastUpdated,surveyLastModified,surveyScheme,cohort.cohortGuid,file.id,file.name,...,Unnamed: 0,names,scores,logfoldchanges,pvals,pvals_adj,cell_type,form_1,form_2,filename
0,910ae58c-41c3-49fb-8e48-08fb25529a61,5a715df8-b774-4947-a023-5fd9c9d97e19_00000000-...,,0001-01-01T00:00:00Z,2026-03-03T19:20:33.242Z,0001-01-01T00:00:00Z,,,5a715df8-b774-4947-a023-5fd9c9d97e19,/home/workspace/input/3733913762/2025_bgmp/5a7...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,CD44,9.302001,0.051674,1.378264e-20,1.842739e-17,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1.0,CD2,8.935438,0.094797,4.055687e-19,2.711227e-16,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.0,S100A10,7.404135,0.028103,1.320082e-13,5.673898e-11,CD4 Central Memory,Afatinib,Afatinib dimaleate,/home/workspace/input/3733913762/2025_bgmp/5a7...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
83437,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1324.0,SELL,-2.734802,-0.350373,6.241792e-03,6.380187e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83438,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1325.0,AKT3,-2.773036,-0.285096,5.553597e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83439,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1326.0,PRKCQ,-2.781397,-0.195473,5.412555e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...
83440,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,1327.0,RIPOR2,-2.910409,-0.245898,3.609560e-03,6.150609e-01,CD8 Effector Memory,Tofacitinib citrate,Tofacitinib,/home/workspace/input/3733913762/2025_bgmp/5a7...


## Clean up the DEG dataframe

In [5]:
# Remove empty rows
deg_df = deg_df.dropna(subset=["names"])

In [6]:
# Keep only columns needed for GSEA
deg_df = deg_df[[
    "cell_type",
    "form_1",
    "form_2",
    "names",
    "logfoldchanges",
    "pvals",
    "pvals_adj"
]]

In [7]:
# Adding a comparison label column
deg_df["comparison"] = deg_df["form_1"] + "_vs._" + deg_df["form_2"]

# Gene set `Reactome_2022` for Gene Set Enrichment Analysis
- `Reactome_2022` has MANY pathways.

In [8]:
### Add a dict to store results
gsea_res = {}

for cell_type, cell_type_df in deg_df.groupby("cell_type"):
    ### Nested dict to store each drug
    gsea_res[cell_type] = {}

    for comparison, comparison_df in cell_type_df.groupby("comparison"):
        
        # Build a ranked gene list: extract ranked genes (names + logFC)
        gene_rank = comparison_df[["names", "logfoldchanges"]]

        # With GSEA of drug treatments vs. DMSO control, previously filtered genes expressed in at least 30 cells with `sc.pp.calculate_qc_metrics` and `subset_cell.var.n_cells_by_counts()`.
        
        # Build a ranked gene list: Sort the genes from high to low fold changes
        gene_rank = gene_rank.sort_values("logfoldchanges", ascending=False)
        
        # Run prerank GSEA with gp.prerank()
        pre_res = gp.prerank(
            rnk=gene_rank,
            gene_sets = "Reactome_2022",
             min_size = 5, # Parameter so that IL-6 related pathways aren't filtered out anymore;
            # Reactome gene sets are very small and specific, and number of features is reduced already
            # by limited gene panel. Default min_size = 15
            outdir = None, # don't write to disk
            verbose = False # make True to see what's going on behind the scenes
        )

        ### Store the output in the dict
        gsea_res[cell_type][comparison] = pre_res

## View results for each cell type and comparison, in a table

In [9]:
# View dictionary key to see what comparisons were made
gsea_res["CD4 Naive"].keys()

dict_keys(['Afatinib_vs._Afatinib dimaleate', 'Baricitinib_vs._Baricitinib phosphate', 'Canertinib_vs._Canertinib dihydrochloride', 'Erlotinib_vs._Erlotinib hydrochloride', 'Gefitinib_vs._Gefitinib hydrochloride', 'NVP-BSK805_vs._NVP-BSK805 2HCl', 'Ruxolitinib_vs._Ruxolitinib phosphate', 'Tofacitinib citrate_vs._Tofacitinib'])

In [10]:
# View results for each cell type and comparison (8 total comparisons!)
pre_res = gsea_res['CD4 Naive']['Tofacitinib citrate_vs._Tofacitinib']
pre_res.res2d.sort_values('FDR q-val').head(10)

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
5,prerank,Signaling By NOTCH1 R-HSA-1980143,0.747301,1.837361,0.005556,0.097046,0.194,5/12,7.43%,NOTCH1;MYC;MAMLD1;HIF1A;RBX1
12,prerank,Interleukin-21 Signaling R-HSA-9020958,0.806722,1.786172,0.0,0.103005,0.344,3/9,1.84%,STAT1;STAT3;JAK3
1,prerank,Interleukin-4 And Interleukin-13 Signaling R-H...,0.610419,1.908656,0.0,0.105559,0.063,11/37,7.59%,BATF;BCL6;MYC;STAT1;STAT3;JAK3;ICAM1;SOCS1;HIF...
7,prerank,Nuclear Envelope Breakdown R-HSA-2980766,0.79619,1.817004,0.0,0.106467,0.248,5/10,12.62%,PLK1;CDK1;NUP62;NUP107;NUP93
11,prerank,NOTCH1 Intracellular Domain Regulates Transcri...,0.79913,1.791668,0.001957,0.108478,0.321,5/9,7.43%,NOTCH1;MYC;MAMLD1;HIF1A;RBX1
4,prerank,Signaling By ALK R-HSA-201556,0.734921,1.84798,0.005245,0.110856,0.171,6/15,5.75%,MYC;STAT3;JAK3;PRDM1;CD274;HIF1A
10,prerank,Interleukin-7 Signaling R-HSA-1266695,0.786372,1.796718,0.003752,0.118234,0.306,3/10,2.40%,STAT3;JAK3;SOCS1
0,prerank,NGF-stimulated Transcription R-HSA-9031628,-0.826862,-1.93805,0.002257,0.134943,0.1,6/9,8.15%,EGR2;FOS;EGR3;FOSB;ID2;EGR1
13,prerank,Inactivation Of CSF3 (G-CSF) Signaling R-HSA-9...,0.820256,1.755616,0.005682,0.139736,0.463,3/8,2.40%,STAT1;STAT3;SOCS1
2,prerank,Estrogen-dependent Nuclear Events Downstream O...,-0.763292,-1.883851,0.0,0.148687,0.204,4/11,9.66%,FOS;BCL2;MAPK3;PTK2


# Create data frame, data frame to CSV, upload to HISE

In [11]:
# empty list
all_results = []

for cell_type in gsea_res:
    for comparison in gsea_res[cell_type]:
        res_df = gsea_res[cell_type][comparison].res2d  # results in rows with columns like Term, NES, FDR q-val, etc...

        # Add more columns for detail
        res_df["cell_type"] = cell_type
        res_df["comparison"] = comparison
        res_df["gene_set"] = "Reactome"   # Hallmark, KEGG, or Reactome

        # Add to dataframe
        all_results.append(res_df)

# Create a dataframe
final_gsea_df = pd.concat(all_results)

In [12]:
final_gsea_df

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes,cell_type,comparison,gene_set
0,prerank,DNA Double-Strand Break Repair R-HSA-5693532,-0.587951,-1.637877,0.004499,1.0,0.811,11/27,16.23%,CDK2;RAD9A;CHEK1;TIMELESS;POLE2;BLM;PARP2;CLSP...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Reactome
1,prerank,Homology Directed Repair R-HSA-5693538,-0.575046,-1.566067,0.014623,1.0,0.976,10/24,16.23%,CDK2;RAD9A;CHEK1;TIMELESS;POLE2;BLM;PARP2;CLSP...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Reactome
2,prerank,Processing Of DNA Double-Strand Break Ends R-H...,-0.633612,-1.555171,0.013174,1.0,0.989,8/14,16.23%,CDK2;RAD9A;CHEK1;TIMELESS;BLM;CLSPN;TOPBP1;TIPIN,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Reactome
3,prerank,Interferon Alpha/Beta Signaling R-HSA-909733,0.375217,1.534439,0.027397,1.0,0.972,8/30,6.28%,OAS1;IFITM1;IFIT1;IRF4;MX1;IRF8;STAT1;IRF2,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Reactome
4,prerank,HDR Thru Homologous Recombination (HRR) Or Sin...,-0.574052,-1.534422,0.021591,1.0,0.995,9/21,16.23%,CDK2;RAD9A;CHEK1;TIMELESS;POLE2;BLM;CLSPN;TOPB...,CD4 Central Memory,Afatinib_vs._Afatinib dimaleate,Reactome
...,...,...,...,...,...,...,...,...,...,...,...,...,...
805,prerank,Apoptosis R-HSA-109581,-0.127447,-0.434054,1.0,1.0,1.0,23/54,40.00%,DNM1L;TFDP1;PSME2;TP53BP2;PTK2;BID;PSMB7;PSMB8...,Treg,Tofacitinib citrate_vs._Tofacitinib,Reactome
806,prerank,Activation Of BAD And Translocation To Mitocho...,0.21218,0.432175,0.989583,0.996085,1.0,2/6,23.22%,AKT3;AKT2,Treg,Tofacitinib citrate_vs._Tofacitinib,Reactome
807,prerank,Generation Of Second Messenger Molecules R-HSA...,-0.174704,-0.394296,1.0,1.0,1.0,10/10,82.75%,CD4;CD3E;NCK1;CD247;FYB1;ZAP70;EVL;ITK;LCK;CD3D,Treg,Tofacitinib citrate_vs._Tofacitinib,Reactome
808,prerank,KEAP1-NFE2L2 Pathway R-HSA-9755511,-0.128329,-0.372378,0.998342,1.0,1.0,6/28,26.98%,PSME2;KEAP1;PSMB7;PSMB8;NFE2L2;PSME1,Treg,Tofacitinib citrate_vs._Tofacitinib,Reactome


In [13]:
final_gsea_df.to_csv("il6_jak-stat_head_to_head_gseapy_reactome.csv")

In [14]:
uuid_list = [
    '02765813-6130-4fac-8708-f01768aa05b6',
    '06b73fab-62d2-4fc3-8a86-2df960cbc1ea',
    '1aecccab-f62a-4c49-b2fe-ba5777930262',
    '207c5f6c-92ec-4690-a7fd-07b0cbebd9f6',
    '3adc31cf-1ea9-4a7d-bc63-31358cb8a325',
    '45e4bd43-4fae-49df-8974-8d043e395f73',
    '4c13d814-1493-48f8-8021-a819b556e97b',
    '56bc5070-2968-4c6b-8198-4e997c75e4fe',
    '64e7735c-e89b-41ef-a293-a39b9ed5ade9',
    '72755c82-880d-4814-8322-6a81ed07466d',
    '9693f71c-dcb3-4d40-b2ad-74fc2ad147de',
    '9f37360e-0191-418b-ab51-fdb86ab9be09',
    'a3bc4704-5fe3-4a74-bce5-0700689db1f6',
    'af42c180-0218-4918-ae12-0a4f43c4aa1f',
    'b790716c-b2de-4028-ad60-1e5a7b81f05d',
    'bfa4c2f1-69d5-4bcf-a1a6-9beb69dbffc9',
    'd076758f-3d76-49b7-91be-0217eefcdf26',
    'd2f8da49-85eb-4dea-a166-5308bb73de91'
]

file_list = hisepy.cache_files(uuid_list)

In [15]:
# a function to make unique destination strings using the periodic table elements
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
        
    rand_str = '-'.join(rand_el)
    return rand_str

In [16]:
# getting unique title for upload, using UTC time
from datetime import datetime, timezone

# getting the current time and saving it
utc_current = datetime.now(timezone.utc)
print(utc_current)

# listing the files to upload
files_to_upload = [
    "il6_jak-stat_head_to_head_gseapy_reactome.csv"
]

# uploading the files
hisepy.upload_files(
    files = files_to_upload,
    study_space_id = "c8a94b84-b0b7-40a9-980b-81a63ad6e115",
    title = f"GSEApy_head-to-head_Reactome_data_h5ad_{utc_current}",
    input_file_ids = uuid_list,
    #destination is randomly generated elements
    destination = element_id()
)

2026-03-04 08:21:40.166603+00:00
checking if conda environment can compile...
creating temp conda environment...
temp conda environment created successfully, now packing...
Cannot determine the current notebook.
1) /home/workspace/gseapy_head_to_head_Reactome_JW.ipynb
2) /home/workspace/gseapy_head_to_head_Hallmark_JW.ipynb
3) /home/workspace/gseapy_head_to_head_KEGG_JW.ipynb
Please select (1-3) 


 1


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'fb9340c8-a66f-4540-8a4f-c4f45dc2fb1a',
 'ProcessId': 'cfcf80fa-6137-4113-a091-77a8eb125304',
 'WorkflowId': '049ffd5b-99b6-4bab-adc2-c9357e20f41a',
 'FileIds': ['63186dcf-a935-4011-b52c-bfb3db974cfb']}